In [1]:
import sys
import os

# Apuntar al raíz del proyecto para encontrar config.py
ROOT = r"D:\inf_sri_hist3\z_CLUSTER"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

import config
print(f"ROOT      : {config.ROOT}")
print(f"DTA_FILE  : {config.DTA_FILE}")
print(f"PARQUET   : {config.PARQUET_FILE}")
print(f"Universo  : {config.N_UNIVERSE:,} contribuyentes")

ROOT      : D:\inf_sri_hist3\z_CLUSTER
DTA_FILE  : D:\inf_sri_hist3\z_CLUSTER\data\B00_DATASET_ANALITICO_INTEGRADO.dta
PARQUET   : D:\inf_sri_hist3\z_CLUSTER\data\B00_DATASET_ANALITICO_INTEGRADO.parquet
Universo  : 797,161 contribuyentes


In [2]:
import pandas as pd
import pyreadstat
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
print("Librerías cargadas OK")
print(f"pandas    : {pd.__version__}")
print(f"pyarrow   : {pa.__version__}")

Librerías cargadas OK
pandas    : 3.0.3
pyarrow   : 25.0.0


In [3]:
print("Leyendo B00_DATASET_ANALITICO_INTEGRADO.dta ...")
print("(puede tardar 2-5 minutos con 797k filas y 263 variables)\n")

df, meta = pyreadstat.read_dta(config.DTA_FILE)

print(f"Filas     : {len(df):,}")
print(f"Columnas  : {len(df.columns):,}")
print(f"Memoria   : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Leyendo B00_DATASET_ANALITICO_INTEGRADO.dta ...
(puede tardar 2-5 minutos con 797k filas y 263 variables)

Filas     : 797,161
Columnas  : 263
Memoria   : 2.35 GB


In [4]:
# Verificar universo
assert len(df) == config.N_UNIVERSE, \
    f"ERROR: se esperaban {config.N_UNIVERSE:,} filas, hay {len(df):,}"

# Verificar unicidad de ID
assert df["ID"].nunique() == config.N_UNIVERSE, \
    "ERROR: ID no es único en todas las filas"

# Verificar que las variables de clustering existen
vars_faltantes = [v for v in config.CLUSTER_VARS if v not in df.columns]
if vars_faltantes:
    print(f"ADVERTENCIA — variables de clustering no encontradas ({len(vars_faltantes)}):")
    for v in vars_faltantes:
        print(f"  {v}")
else:
    print(f"Todas las variables de clustering presentes ({len(config.CLUSTER_VARS)})")

print("\nIntegridad verificada OK")

Todas las variables de clustering presentes (80)

Integridad verificada OK


In [5]:
print("Guardando como Parquet...")

df.to_parquet(
    config.PARQUET_FILE,
    engine="pyarrow",
    compression="snappy",
    index=False
)

size_dta     = config.DTA_FILE.stat().st_size / 1e6
size_parquet = config.PARQUET_FILE.stat().st_size / 1e6

print(f"Guardado en: {config.PARQUET_FILE}")
print(f"Tamaño .dta     : {size_dta:.1f} MB")
print(f"Tamaño .parquet : {size_parquet:.1f} MB")
print(f"Reducción       : {(1 - size_parquet/size_dta)*100:.1f}%")

Guardando como Parquet...
Guardado en: D:\inf_sri_hist3\z_CLUSTER\data\B00_DATASET_ANALITICO_INTEGRADO.parquet
Tamaño .dta     : 1007.8 MB
Tamaño .parquet : 404.2 MB
Reducción       : 59.9%


In [6]:
print("Verificando lectura desde Parquet...")

df2 = pd.read_parquet(config.PARQUET_FILE)

assert df2.shape == df.shape, \
    f"ERROR: shape no coincide. dta={df.shape}, parquet={df2.shape}"

print(f"Shape Parquet : {df2.shape}")
print("Lectura desde Parquet verificada OK")
print("\nA partir de aquí todos los notebooks cargan desde Parquet.")

# Liberar df original de memoria
del df2

Verificando lectura desde Parquet...
Shape Parquet : (797161, 263)
Lectura desde Parquet verificada OK

A partir de aquí todos los notebooks cargan desde Parquet.


In [7]:
print(f"F01 : {len(config.F01_VARS)}")
print(f"F02 : {len(config.F02_VARS)}")
print(f"F03 : {len(config.F03_VARS)}")
print(f"F04 : {len(config.F04_VARS)}")
print(f"F05 : {len(config.F05_VARS)}")
print(f"F06 : {len(config.F06_VARS)}")
print(f"F07 : {len(config.F07_VARS)}")
print(f"TOTAL: {len(config.CLUSTER_VARS)}")

F01 : 42
F02 : 6
F03 : 8
F04 : 7
F05 : 7
F06 : 6
F07 : 4
TOTAL: 80


### Verificaciones adicionales

In [8]:
import pandas as pd

# Cargar B00 que ya tiene todos los IDs integrados
B00 = pd.read_parquet(
    r"D:\inf_sri_hist3\z_CLUSTER\data\B00_DATASET_ANALITICO_INTEGRADO.parquet",
    columns=["ID"]
)

# Cargar F01 desde su perfil analítico
f01 = pd.read_parquet(
    r"D:\inf_sri_hist3\z_CLUSTER\data\B01_MATRIZ_CLUSTERING.parquet",
    columns=["ID"]
) if os.path.exists(r"D:\inf_sri_hist3\z_CLUSTER\data\B01_MATRIZ_CLUSTERING.parquet") else None

print(f"Total IDs en B00: {len(B00)}")
if f01 is not None:
    print(f"Total IDs en B01: {len(f01)}")
    no_en_b01 = set(B00["ID"]) - set(f01["ID"])
    print(f"IDs en B00 pero no en B01: {len(no_en_b01)}")

Total IDs en B00: 797161
Total IDs en B01: 797161
IDs en B00 pero no en B01: 0
